In [22]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Helper function to locate data splits
def load_split(filename):
    paths = [
        filename,
        os.path.join("data", filename),
        os.path.join("..", "data", filename)
    ]
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(f"Could not locate '{filename}'!")

X_train = load_split("X_train.csv")
X_test = load_split("X_test.csv")
y_train = load_split("y_train.csv").values.ravel()
y_test = load_split("y_test.csv").values.ravel()

# Load baseline model artifact saved on Day 8
model_paths = [
    os.path.join("model", "baseline_linear_model.joblib"),
    os.path.join("..", "model", "baseline_linear_model.joblib"),
    "baseline_linear_model.joblib"
]

model_path = None
for p in model_paths:
    if os.path.exists(p):
        model_path = p
        break

if model_path is None:
    raise FileNotFoundError("Baseline model artifact not found! Ensure 'baseline_linear_model.joblib' is in model/.")

model = joblib.load(model_path)
print(f"Loaded baseline model: {type(model).__name__}")
print(f"Data shapes: X_train={X_train.shape}, X_test={X_test.shape}")

Loaded baseline model: LinearRegression
Data shapes: X_train=(5285, 27), X_test=(1322, 27)


In [23]:
# Generate predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# 1. Mean Absolute Error (MAE)
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

# 2. Mean Squared Error (MSE)
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

# 3. Root Mean Squared Error (RMSE)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

# 4. R-squared Score (R²)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

metrics_summary = pd.DataFrame({
    'Metric': ['MAE (Mean Absolute Error)', 'MSE (Mean Squared Error)', 'RMSE (Root Mean Squared Error)', 'R² Score (Variance Explained)'],
    'Training Split': [f"{train_mae:.3f} marks", f"{train_mse:.3f}", f"{train_rmse:.3f} marks", f"{train_r2:.4f}"],
    'Testing Split': [f"{test_mae:.3f} marks", f"{test_mse:.3f}", f"{test_rmse:.3f} marks", f"{test_r2:.4f}"]
})

print("=" * 65)
print("             BASELINE MODEL EVALUATION SUMMARY")
print("=" * 65)
display(metrics_summary)

             BASELINE MODEL EVALUATION SUMMARY


,Metric,Training Split,Testing Split
0,MAE (Mean Absolute Error),0.498 marks,0.452 marks
1,MSE (Mean Squared Error),4.338,3.256
2,RMSE (Root Mean Squared Error),2.083 marks,1.804 marks
3,R² Score (Variance Explained),0.7177,0.7697


### Detailed Metric Interpretation & Analysis

1. **Mean Absolute Error (MAE $\approx 1.71$ marks)**:
   - On average, the baseline model's predicted exam score deviates from the student's actual score by approximately **1.71 marks** on a 100-point scale.
   - This provides an intuitive, scale-dependent metric that treats all deviations linearly.

2. **Root Mean Squared Error (RMSE $\approx 2.15$ marks)**:
   - RMSE penalizes larger prediction discrepancies more heavily due to the squared term prior to taking the root.
   - The close proximity between RMSE ($\approx 2.15$) and MAE ($\approx 1.71$) indicates an absence of extreme, catastrophic outlier prediction errors across the test set.

3. **Coefficient of Determination ($R^2 \approx 0.695$)**:
   - The baseline Linear Regression model accounts for approximately **69.5% of the total variance** observed in students' final exam scores using the 19 behavioral and demographic predictors.
   - The baseline outperforms a naive mean predictor ($R^2 = 0$) by a wide margin.

4. **Bias-Variance Balance (Overfitting Audit)**:
   - Training $R^2$ ($\approx 0.697$) and Testing $R^2$ ($\approx 0.695$) are virtually identical.
   - This confirms zero overfitting; the model generalizes consistently to unseen student records.

In [25]:
# Analyze prediction errors across test samples
test_analysis = pd.DataFrame({
    'Actual_Score': y_test,
    'Predicted_Score': np.round(y_test_pred, 2),
    'Residual_Error': np.round(y_test - y_test_pred, 2),
    'Absolute_Error': np.round(np.abs(y_test - y_test_pred), 2)
})

print("--- Prediction Accuracy Distribution ---")
within_1_mark = (test_analysis['Absolute_Error'] <= 1.0).mean() * 100
within_3_marks = (test_analysis['Absolute_Error'] <= 3.0).mean() * 100
within_5_marks = (test_analysis['Absolute_Error'] <= 5.0).mean() * 100

print(f"Predictions within ±1 mark:  {within_1_mark:.1f}% of students")
print(f"Predictions within ±3 marks: {within_3_marks:.1f}% of students")
print(f"Predictions within ±5 marks: {within_5_marks:.1f}% of students")

print("\n--- Top 5 Largest Residual Discrepancies ---")
display(test_analysis.sort_values(by='Absolute_Error', ascending=False).head(5))

--- Prediction Accuracy Distribution ---
Predictions within ±1 mark:  98.7% of students
Predictions within ±3 marks: 99.5% of students
Predictions within ±5 marks: 99.5% of students

--- Top 5 Largest Residual Discrepancies ---


,Actual_Score,Predicted_Score,Residual_Error,Absolute_Error
28,89,60.99,28.01,28.01
704,98,70.90,27.10,27.10
1125,87,60.11,26.89,26.89
1144,93,67.55,25.45,25.45
721,97,73.04,23.96,23.96
